## Importanto Bibliotecas

In [1]:
import os, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from datetime import datetime, timedelta

import nilmtk
from nilmtk.utils import print_dict
from nilmtk import DataSet, MeterGroup

## Iniciando a Base de Dados

In [2]:
ukdale = DataSet('../ukdale.h5')

# CASA 2

-   ## Definindo Parâmetros

In [ ]:
#ALTERE AQUI PARA DEFINIR O APARELHO A SER GERADO AS IMAGENS
APARELHO_ATUAL = 'dish washer'

#DEFINE OS PARAMETROS PARA A GERACAO DAS IMAGENS
TAXA_AMOSTRAGEM = 6

#DEFINE AS DATAS INICIAIS E FINAIS COMO STRINGS
DATA_INICIAL_CASA2 = '2013-06-20T00:00:00'
DATA_FINAL_CASA2 = '2013-07-01T00:00:00'

# Convertendo as datas
DATA_INICIAL_CASA2 = datetime.strptime(DATA_INICIAL_CASA2, '%Y-%m-%dT%H:%M:%S')
DATA_FINAL_CASA2 = datetime.strptime(DATA_FINAL_CASA2, '%Y-%m-%dT%H:%M:%S')

#DEFINE O INTERVALO DE TEMPO
ukdale.set_window(start=DATA_INICIAL_CASA2, end=DATA_FINAL_CASA2)

#ESCOLHE A CASA 1 E OBTEM OS MEDIDORES
elec = ukdale.buildings[2].elec
mains = elec.mains()


-   ## Definindo Aparelhos

In [ ]:
#DEFINE OS APARELHOS E SEUS RESPECTIVOS LIMIARES
APARELHOS = ['fridge', 'kettle', 'washing machine', 'dish washer', 'microwave']
LIMIARES = [50, 200, 20, 20, 100]
RATIOS = [0.5, 0.2, 0.5, 0.5, 0.2]
TAMANHO_JANELA = [600, 300, 600, 600, 300] 
AMOSTRAGENS_POR_JANELA = [janela // TAXA_AMOSTRAGEM for janela in TAMANHO_JANELA]
PASSOS = [amostragem // 10 for amostragem in AMOSTRAGENS_POR_JANELA]

#ENCONTRA QUAL O INDICE DO APARELHO SELECIONADO
indice_aparelho = APARELHOS.index(APARELHO_ATUAL)

#OBTEM OS MEDIDORES DO APARELHO SELECIONADO
grupo_aparelho = (elec.select_using_appliances(type=APARELHOS[indice_aparelho]))
medidor_aparelho = (grupo_aparelho.meters[0])

In [5]:
#DEFINE A FUNCAO GAF (Gramian Angular Fields), METODO DE GERACAO DE IMAGENS QUE SERA USADO
def gaf(signal, noise_std=0.01):

    signal = np.asarray(signal)

    # Normalização para [-1, 1]
    min_v, max_v = np.min(signal), np.max(signal)
    if max_v - min_v == 0:
        return None

    signal = (signal - min_v) / (max_v - min_v)
    signal = signal * 2 - 1
    signal = np.clip(signal, -1, 1)
    
    phi = np.arccos(signal)
    return np.cos(phi[:, None] + phi[None, :])


In [6]:
#OBTEM AS MEDIDAS DE POTENCIA ATIVA DO MEDIDOR CENTRAL
mains_P = next(mains.load(
    physical_quantity='power',
    ac_type='active',
    sample_period=TAXA_AMOSTRAGEM
))

#OBTEM AS MEDIDAS DE POTENCIA APARENTE DO MEDIDOR CENTRAL
mains_S = next(mains.load(
    physical_quantity='power',
    ac_type='apparent',
    sample_period=TAXA_AMOSTRAGEM
))

#OBTEM AS MEDIDAS DE TENSAO DO MEDIDOR CENTRAL
mains_V = next(mains.load(
    physical_quantity='voltage',
    sample_period=TAXA_AMOSTRAGEM
))


df_aparelho = (next(medidor_aparelho.load(
    physical_quantity='power',
    ac_type='active',
    sample_period=TAXA_AMOSTRAGEM
)))


In [7]:
# Remove MultiIndex se existir
if isinstance(df_aparelho.columns, pd.MultiIndex):
    df_aparelho.columns = df_aparelho.columns.get_level_values(-1)

# Constrói o dataframe diretamente
df = pd.concat(
    [mains_P, mains_S, mains_V, df_aparelho],
    axis=1
)

df = df.dropna()

# Opcional (RECOMENDADO): força nomes claros
df.columns = ["P", "S", "V", "P_sub"]


In [8]:
#CRIA AS PASTAS RESPECTIVAS PARA CADA UM DOS APARELHOS

shutil.rmtree(f"gaf_images/teste/{APARELHO_ATUAL}", ignore_errors=True)     #APAGA A PASTA CASO JA EXISTA
os.makedirs(f"gaf_images/teste/{APARELHO_ATUAL}/on", exist_ok=True)
os.makedirs(f"gaf_images/teste/{APARELHO_ATUAL}/off", exist_ok=True)


-   ## Criando as Imagens

In [9]:
import random

# =============================
# SEGUNDA PASSADA → ARMAZENAR TODAS JANELAS
# =============================

buffer_P, buffer_Q, buffer_I, buffer_sub = [], [], [], []

lista_on = []
lista_off = []

print(f"========== {APARELHO_ATUAL} ==========")

for _, row in df.iterrows():

    P = row.iloc[0]
    S = row.iloc[1]
    V = row.iloc[2]
    P_sub = row.iloc[3]

    I = S / V if V > 0 else 0
    Q = np.sqrt(max(S**2 - P**2, 0))

    buffer_P.append(P)
    buffer_Q.append(Q)
    buffer_I.append(I)
    buffer_sub.append(P_sub)

    if len(buffer_P) == AMOSTRAGENS_POR_JANELA[indice_aparelho]:

        on_ratio = np.mean(np.array(buffer_sub) > LIMIARES[indice_aparelho])
        label = "on" if on_ratio >= RATIOS[indice_aparelho] else "off"

        # Salva a janela em memória (não gera imagem ainda)
        janela = (
            buffer_P.copy(),
            buffer_Q.copy(),
            buffer_I.copy()
        )

        if label == "on":
            lista_on.append(janela)
        else:
            lista_off.append(janela)

        buffer_P = buffer_P[PASSOS[indice_aparelho]:]
        buffer_Q = buffer_Q[PASSOS[indice_aparelho]:]
        buffer_I = buffer_I[PASSOS[indice_aparelho]:]
        buffer_sub = buffer_sub[PASSOS[indice_aparelho]:]


# =============================
# BALANCEAMENTO ALEATÓRIO
# =============================

random.shuffle(lista_on)
random.shuffle(lista_off)

MAX_POR_CLASSE = min(len(lista_on), len(lista_off))

lista_on = lista_on[:MAX_POR_CLASSE]
lista_off = lista_off[:MAX_POR_CLASSE]

print(f"Total ON final: {len(lista_on)}")
print(f"Total OFF final: {len(lista_off)}")

idx = 0

def salvar_imagens(lista, label):

    qtdImagens = len(lista)
    Porcentagem = 0
    contador = 0

    global idx

    for i, (buffer_P, buffer_Q, buffer_I) in enumerate(lista):

        gaf_P = gaf(buffer_P)
        gaf_Q = gaf(buffer_Q)
        gaf_I = gaf(buffer_I)

        img = np.stack([gaf_P, gaf_Q, gaf_I], axis=-1)
        img = (img + 1) / 2
        img = np.clip(img, 0, 1)

        contador += 1
        progresso = (contador / qtdImagens) * 100
        if progresso >= Porcentagem + 10:
            Porcentagem += 10
            print(f'Total de imagens \"{label}\": {contador}, Porcentagem: {Porcentagem}%')

        plt.imsave(
            f"gaf_images/teste/{APARELHOS[indice_aparelho]}/{label}/gaf_{idx}.png",
            img
        )

        idx += 1


salvar_imagens(lista_on, "on")
salvar_imagens(lista_off, "off")


========== dish washer ==========
Total ON final: 484
Total OFF final: 484
Total de imagens "on": 49, Porcentagem: 10%
Total de imagens "on": 97, Porcentagem: 20%
Total de imagens "on": 146, Porcentagem: 30%
Total de imagens "on": 194, Porcentagem: 40%
Total de imagens "on": 242, Porcentagem: 50%
Total de imagens "on": 291, Porcentagem: 60%
Total de imagens "on": 339, Porcentagem: 70%
Total de imagens "on": 388, Porcentagem: 80%
Total de imagens "on": 436, Porcentagem: 90%
Total de imagens "on": 484, Porcentagem: 100%
Total de imagens "off": 49, Porcentagem: 10%
Total de imagens "off": 97, Porcentagem: 20%
Total de imagens "off": 146, Porcentagem: 30%
Total de imagens "off": 194, Porcentagem: 40%
Total de imagens "off": 242, Porcentagem: 50%
Total de imagens "off": 291, Porcentagem: 60%
Total de imagens "off": 339, Porcentagem: 70%
Total de imagens "off": 388, Porcentagem: 80%
Total de imagens "off": 436, Porcentagem: 90%
Total de imagens "off": 484, Porcentagem: 100%
